In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df1 = pd.read_csv(r"C:\Users\hi\Desktop\Subcounty dataset\datasets\Distribution of Population.csv")
df2 = pd.read_csv(r"C:\Users\hi\Desktop\Subcounty dataset\datasets\ke_subcounty_with_coords.csv")
df3 = pd.read_csv(r"C:\Users\hi\Desktop\Subcounty dataset\datasets\master_risk_df.csv")
df4 = pd.read_csv(r"C:\Users\hi\Desktop\Subcounty dataset\datasets\Subcounty Hospital Count.csv")

In [3]:
df1 = df1[['County', 'Subcounty', 'Total', 'Male', 'Female', 'SQ.Km']]
df1.columns

Index(['County', 'Subcounty', 'Total', 'Male', 'Female', 'SQ.Km'], dtype='object')

In [6]:
df_merged.head()

NameError: name 'df_merged' is not defined

In [4]:
# Strip accidental whitespace and convert to lowercase/uppercase consistency
for df in [df1, df2, df3, df4]:
    df['Subcounty'] = df['Subcounty'].astype(str).str.strip().str.title()

# The Complete Pre-Processing Pipeline

In [5]:
import re

def clean_subcounty_names(df, col_name='Subcounty'):
    # 1. Convert to string and strip whitespace
    df[col_name] = df[col_name].astype(str).str.strip()
    
    # 2. Remove variations of "Sub-County" or "Sub County" case-insensitively
    df[col_name] = df[col_name].str.replace(r'(?i)\s*sub[- ]*county', '', regex=True)
    
    # 3. Standardize to Title Case (e.g., KISUMU CENTRAL -> Kisumu Central)
    df[col_name] = df[col_name].str.title()
    
    # 4. Strip out stray double spaces caused by the replacements
    df[col_name] = df[col_name].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df

# Apply the cleaning function to every dataframe
df1 = clean_subcounty_names(df1)
df2 = clean_subcounty_names(df2)
df3 = clean_subcounty_names(df3)
df4 = clean_subcounty_names(df4)

# Handle known hardcoded exceptions to match your df2 parent standard
manual_fixes = {
    'Olkalou': 'Ol Kalou',
    'Homa Bay Town': 'Homa Bay'
}
for df in [df1, df3, df4]:
    df['Subcounty'] = df['Subcounty'].replace(manual_fixes)

# Now execute your parent-first left merge sequence safely
dataframes = [df2, df1, df3, df4]
df_merged = reduce(lambda left, right: pd.merge(left, right, on="Subcounty", how="left"), dataframes)

NameError: name 'reduce' is not defined

In [ ]:
# Filter down to just the rows with NaN values
df_nan_rows = df_risk[df_risk.isnull().any(axis=1)]

print(f"--- Found {len(df_nan_rows)} Subcounties with Missing Values ---\n")

# Loop and print out the missing columns for each subcounty
for index, row in df_nan_rows.iterrows():
    # Identify which columns are null for this specific row
    missing_cols = row[row.isnull()].index.tolist()
    
    print(f"Subcounty: {row['Subcounty']} ({row['County']})")
    print(f"❌ Missing Columns: {missing_cols}")
    print("-" * 50)

In [ ]:
# Columns you want to display for context
display_cols = ['Subcounty', 'County', 'Total', 'longitude', 'latitude', 'raw_healthcare_capacity']

# Display the rows that have NaNs, showing only the context columns
df_risk[df_risk.isnull().any(axis=1)][display_cols]

In [ ]:
import pandas as pd
import numpy as np

# 1. Define lists of critical columns based on your datasets
health_cols = ['raw_healthcare_capacity', 'Health Centre', 'Dispensary', 'District Hospital']
spatial_cols = ['geometry', 'road_count', 'dist_to_border_km', 'hospital_count']
demo_cols = ['County', 'Total', 'SQ.Km']

# 2. Filter down to only rows that have at least one NaN value
df_nan_rows = df_risk[df_risk.isnull().any(axis=1)].copy()

# 3. Create a helper function to profile what each row is missing
def profile_missing_row(row):
    missing_health = any(pd.isna(row[col]) for col in health_cols if col in row)
    missing_spatial = any(pd.isna(row[col]) for col in spatial_cols if col in row)
    missing_demo = any(pd.isna(row[col]) for col in demo_cols if col in row)
    
    if missing_health and missing_spatial and missing_demo:
        return "Profile A: Complete Merge Failure (Missing All Datasets)"
    elif missing_health and not missing_spatial and not missing_demo:
        return "Profile B: Missing Health Metrics Only"
    elif missing_spatial and not missing_health and not missing_demo:
        return "Profile C: Missing Spatial/GIS Metrics Only"
    elif missing_demo and not missing_health and not missing_spatial:
        return "Profile D: Missing Demographic Metrics Only"
    elif missing_health and missing_spatial and not missing_demo:
        return "Profile E: Missing Health + Spatial Metrics"
    elif missing_health and missing_demo and not missing_spatial:
        return "Profile F: Missing Health + Demographic Metrics"
    elif missing_demo and missing_spatial and not missing_health:
        return "Profile G: Missing Demographic + Spatial Metrics"
    else:
        return "Profile H: Miscellaneous Column Gaps"

# 4. Apply profiling to the missing data rows
df_nan_rows['Missing_Profile'] = df_nan_rows.apply(profile_missing_row, axis=1)

# 5. Group by the Profile to create the complete summary dataframe
df_missing_summary = df_nan_rows.groupby('Missing_Profile').agg(
    Total_Affected_Subcounties=('Subcounty', 'count'),
    Subcounty_List=('Subcounty', lambda x: ", ".join(x.astype(str)))
).reset_index()

# 6. View your complete summary table
pd.set_option('display.max_colwidth', None)  # Ensures text isn't cut off in display
print(df_missing_summary)

In [ ]:
df_missing_summary.head()

In [ ]:
pd.set_option('display.max_columns',None)
df_merged.info()

In [ ]:
import re

def clean_subcounty_names(df, col_name='Subcounty'):
    # 1. Convert to string and strip whitespace
    df[col_name] = df[col_name].astype(str).str.strip()
    
    # 2. Remove variations of "Sub-County" or "Sub County" case-insensitively
    df[col_name] = df[col_name].str.replace(r'(?i)\s*sub[- ]*county', '', regex=True)
    
    # 3. Standardize to Title Case (e.g., KISUMU CENTRAL -> Kisumu Central)
    df[col_name] = df[col_name].str.title()
    
    # 4. Strip out stray double spaces caused by the replacements
    df[col_name] = df[col_name].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df

# Apply the cleaning function to every dataframe
df1 = clean_subcounty_names(df1)
df2 = clean_subcounty_names(df2)
df3 = clean_subcounty_names(df3)
df4 = clean_subcounty_names(df4)

# Handle known hardcoded exceptions to match your df2 parent standard
manual_fixes = {
    'Olkalou': 'Ol Kalou',
    'Homa Bay Town': 'Homa Bay'
}
for df in [df1, df3, df4]:
    df['Subcounty'] = df['Subcounty'].replace(manual_fixes)

# Now execute your parent-first left merge sequence safely
dataframes = [df2, df1, df3, df4]
df_merged = reduce(lambda left, right: pd.merge(left, right, on="Subcounty", how="left"), dataframes)

In [ ]:
from functools import reduce
import pandas as pd

# 1. Clean the 'Subcounty' column across all dataframes to ensure perfect matching
for df in [df1, df2, df3, df4]:
    df['Subcounty'] = df['Subcounty'].astype(str).str.strip().str.title()

# 2. Put df2 FIRST in the list so it acts as the parent/left dataframe
dataframes = [df2, df1, df3, df4]

# 3. Merge them using how='left'
df_merged = reduce(
    lambda left, right: pd.merge(left, right, on="Subcounty", how="left"),
    dataframes,
)

# 4. View the finalized master dataframe
print(df_merged.head())

In [ ]:
df_merged.info()

In [ ]:
df_merged.head()

In [ ]:
df_merged.to_csv('df_merged.csv', index=False)

In [ ]:
risk_columns = [
    # Identifiers
    'Subcounty', 'County', 'longitude', 'latitude', 'geometry',
    
    # Vulnerability/Density
    'Total', 'SQ.Km',
    
    # Transmission / Proximity
    'road_count', 'dist_to_border_km', 'dist_to_laikipia_airbase_km',
    'dist_to_busia_km', 'dist_to_malaba_km', 'dist_to_lwakhakha_km', 'dist_to_suam_km',
    
    # Capacity / Mitigation
    'raw_healthcare_capacity', 'hospital_count', 'Health Centre', 'Dispensary', 
    'District Hospital', 'Provincial General Hospital', 'National Referral Hospital',
    'Laboratory (Stand-alone)'
]

# Create your dedicated risk modeling dataframe
df_risk = df_merged[risk_columns].copy()

In [ ]:
df_risk[df_risk.Subcounty == 'Nandi East']

In [ ]:
# Filter down to just the rows with NaN values
df_nan_rows = df_risk[df_risk.isnull().any(axis=1)]

print(f"--- Found {len(df_nan_rows)} Subcounties with Missing Values ---\n")

# Loop and print out the missing columns for each subcounty
for index, row in df_nan_rows.iterrows():
    # Identify which columns are null for this specific row
    missing_cols = row[row.isnull()].index.tolist()
    
    print(f"Subcounty: {row['Subcounty']} ({row['County']})")
    print(f"❌ Missing Columns: {missing_cols}")
    print("-" * 50)